In [ ]:
##task 1

In [112]:
import csv
import json

with open("data/appointments.csv", newline="") as f:
    reader = csv.DictReader(f)
    print(reader.fieldnames)
    rows = list(reader)

print(len(rows))

['age', 'fee_inr', 'prior_no_shows', 'distance_km', 'booking_channel', 'scheduled_date', 'appointment_date', 'no_show']
1020


In [113]:
print(rows[0])

{'age': '48.0', 'fee_inr': '1320', 'prior_no_shows': '4', 'distance_km': '13.82', 'booking_channel': 'Mobile App', 'scheduled_date': '2023-01-01 00:00:00', 'appointment_date': '2023-01-04', 'no_show': 'No'}


In [114]:
def parse_row(row):
    new = dict(row)
    new["age"] = int(float(row["age"]))
    new["fee_inr"] = int(row["fee_inr"])
    new["prior_no_shows"] = int(row["prior_no_shows"])
    new["distance_km"] = float(row["distance_km"])
    return new

print(parse_row(rows[0]))

{'age': 48, 'fee_inr': 1320, 'prior_no_shows': 4, 'distance_km': 13.82, 'booking_channel': 'Mobile App', 'scheduled_date': '2023-01-01 00:00:00', 'appointment_date': '2023-01-04', 'no_show': 'No'}


In [115]:
accepted, rejected = [], []

for row in rows:
    try:
        accepted.append(parse_row(row))
    except ValueError:
        rejected.append(row)

print("read:", len(rows))
print("accepted:", len(accepted))
print("rejected:", len(rejected))

read: 1020
accepted: 988
rejected: 32


In [116]:
from collections import Counter

fails = Counter()
examples = {}
for row in rows:
    for col, func in [("age", int), ("fee_inr", int),
                      ("prior_no_shows", int), ("distance_km", float)]:
        try:
            func(row[col])
        except ValueError:
            fails[col] += 1
            examples.setdefault(col, row[col])

print(fails)
print(examples)

Counter({'age': 1020, 'distance_km': 20})
{'age': '48.0', 'distance_km': ''}


In [117]:
with open("rejected_rows.json", "w") as f:
    json.dump(rejected, f, indent=2)

print("saved", len(rejected), "rows")

saved 32 rows


In [118]:
blank_age = sum(1 for r in rejected if r["age"] == "")
blank_dist = sum(1 for r in rejected if r["distance_km"] == "")
print(blank_age, blank_dist)

12 20


In [119]:
too_old = [r for r in accepted if r["age"] > 120]
print(len(too_old))
print(sorted({r["age"] for r in too_old}))

5
[250]


The age-250 rows were accepted because `int("250")` converts without error, so the code only checked the type (is it an integer?) and not the validity (is it a plausible human age?).

In [ ]:
##task 2

In [120]:
import pandas as pd

df = pd.read_csv("data/appointments.csv")

In [121]:
print(df.shape)

(1020, 8)


In [122]:
print(df.shape)
df.info()
print(df.isnull().sum())
print(df.duplicated().sum())

(1020, 8)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   age               1008 non-null   float64
 1   fee_inr           1020 non-null   int64  
 2   prior_no_shows    1020 non-null   int64  
 3   distance_km       1000 non-null   float64
 4   booking_channel   973 non-null    object 
 5   scheduled_date    1020 non-null   object 
 6   appointment_date  1020 non-null   object 
 7   no_show           1020 non-null   object 
dtypes: float64(2), int64(2), object(4)
memory usage: 63.9+ KB
age                 12
fee_inr              0
prior_no_shows       0
distance_km         20
booking_channel     47
scheduled_date       0
appointment_date     0
no_show              0
dtype: int64
0


In [123]:
print(df["booking_channel"].nunique())
print(df["booking_channel"].value_counts(dropna=False))

6
booking_channel
Mobile App     427
Website        195
Call Center    192
Walk-in         57
WEBSITE         54
mobile app      48
NaN             47
Name: count, dtype: int64


In [124]:
df = df.drop_duplicates()
print(len(df))

1020


In [125]:
df["booking_channel"] = df["booking_channel"].str.strip().str.title()
print(df["booking_channel"].nunique())
print(df["booking_channel"].unique())
print(len(df))

4
['Mobile App' nan 'Call Center' 'Website' 'Walk-In']
1020


In [126]:
df["appointment_date"] = pd.to_datetime(df["appointment_date"], errors="coerce")
df["scheduled_date"] = pd.to_datetime(df["scheduled_date"])
print(len(df))
print(df["appointment_date"].isnull().sum())
print(df.dtypes)

1020
0
age                        float64
fee_inr                      int64
prior_no_shows               int64
distance_km                float64
booking_channel             object
scheduled_date      datetime64[ns]
appointment_date    datetime64[ns]
no_show                     object
dtype: object


In [127]:
df = df.dropna(subset=["appointment_date", "no_show"])
print(len(df))

1020


In [128]:
df.to_csv("cleaned_pass1.csv", index=False)

In [ ]:
##task 3

In [129]:
import numpy as np
print(len(df))

1020


In [130]:
mask = (df["age"] < 1) | (df["age"] > 100)
print("changed:", mask.sum())
print(sorted(df.loc[mask, "age"].unique()))

df.loc[mask, "age"] = np.nan
print("missing ages:", df["age"].isnull().sum())

changed: 15
[np.float64(0.0), np.float64(250.0)]
missing ages: 27


In [131]:
age_median = df["age"].median()
print("age median:", age_median)
df["age"] = df["age"].fillna(age_median).round().astype(int)

dist_median = df["distance_km"].median()
print("distance median:", dist_median)
df["distance_km"] = df["distance_km"].fillna(dist_median)

print(df[["age", "distance_km"]].isnull().sum())

age median: 49.0
distance median: 25.285
age            0
distance_km    0
dtype: int64


In [132]:
for col in ["distance_km", "fee_inr"]:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    upper = q3 + 1.5 * iqr
    lower = q1 - 1.5 * iqr
    n_capped = ((df[col] > upper) | (df[col] < lower)).sum()
    print(col, "Q1:", q1, "Q3:", q3, "upper:", upper, "lower:", lower, "capped:", n_capped)
    df[col] = np.clip(df[col], lower, upper)

distance_km Q1: 12.72 Q3: 36.97 upper: 73.345 lower: -23.655 capped: 0
fee_inr Q1: 511.25 Q3: 1495.0 upper: 2970.625 lower: -964.375 capped: 0


In [133]:
df["distance_km"] = df["distance_km"].round(1)
df["fee_inr"] = df["fee_inr"].round().astype(int)

print(df["distance_km"].max(), df["fee_inr"].max())
print(df.dtypes)
print(df.isnull().sum())

50.0 2000
age                          int64
fee_inr                      int64
prior_no_shows               int64
distance_km                float64
booking_channel             object
scheduled_date      datetime64[ns]
appointment_date    datetime64[ns]
no_show                     object
dtype: object
age                  0
fee_inr              0
prior_no_shows       0
distance_km          0
booking_channel     47
scheduled_date       0
appointment_date     0
no_show              0
dtype: int64


My file contained 15 invalid ages (nine 0s and six 250s), not the brief’s 5, so after step (a) 27 ages were missing and the age median was 49. The distance median was 25.285 and the IQR fences were too wide to capture any value in distance_km or fee_inr, so 0 values were capped. The brief’s figures describe a different version of the dataset.

In [134]:
print(df.shape)
print(df.isnull().sum())
print(df.dtypes)
df.to_csv("cleaned_pass1.csv", index=False)   # backup copy

(1020, 8)
age                  0
fee_inr              0
prior_no_shows       0
distance_km          0
booking_channel     47
scheduled_date       0
appointment_date     0
no_show              0
dtype: int64
age                          int64
fee_inr                      int64
prior_no_shows               int64
distance_km                float64
booking_channel             object
scheduled_date      datetime64[ns]
appointment_date    datetime64[ns]
no_show                     object
dtype: object
